# Store Performance Validation

**Owner:** Mannan  
**Assigned reviewer:** Maheshwar  
**Run after:** `03_gold_eda.ipynb`

Recomputes store orders, customers, revenue, and AOV from orders_gold and reconciles them with store_performance_gold.

This notebook is an owner-specific PySpark contribution. The owner must run it personally, inspect the displayed result, understand every assertion, and commit it from their own GitHub account.


## 1. Load the validated project tables


In [ ]:
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "analytics"
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

stores = spark.table(f"{CATALOG}.{SCHEMA}.stores_silver")
orders_gold = spark.table(f"{CATALOG}.{SCHEMA}.orders_gold")
store_performance = spark.table(f"{CATALOG}.{SCHEMA}.store_performance_gold")

print(f"stores_silver: {stores.count():,}")
print(f"orders_gold: {orders_gold.count():,}")
print(f"store_performance_gold: {store_performance.count():,}")


## 2. Run owner-specific reconciliation and integrity checks


In [ ]:
# Cleaned stores and store-performance rows must each have unique StoreID values.
assert stores.select("StoreID").distinct().count() == stores.count()
assert store_performance.select("StoreID").distinct().count() == store_performance.count()

# Every store-performance row must refer to a cleaned store.
assert store_performance.join(
    stores.select("StoreID"), "StoreID", "left_anti"
).count() == 0

# Recompute store metrics from one-row-per-order Gold data.
expected_store_metrics = orders_gold.groupBy("StoreID", "StoreName").agg(
    F.countDistinct("OrderID").alias("ExpectedOrders"),
    F.countDistinct("CustomerID").alias("ExpectedCustomers"),
    F.round(F.sum("NetRevenue"), 2).alias("ExpectedRevenue"),
    F.round(F.avg("NetRevenue"), 2).alias("ExpectedAOV"),
)

store_reconciliation = store_performance.alias("actual").join(
    expected_store_metrics.alias("expected"), ["StoreID", "StoreName"], "full"
).filter(
    (F.coalesce(F.col("actual.CompletedOrders"), F.lit(-1))
     != F.coalesce(F.col("expected.ExpectedOrders"), F.lit(-2)))
    | (F.coalesce(F.col("actual.UniqueCustomers"), F.lit(-1))
       != F.coalesce(F.col("expected.ExpectedCustomers"), F.lit(-2)))
    | (F.abs(F.coalesce(F.col("actual.NetRevenue"), F.lit(-1.0))
             - F.coalesce(F.col("expected.ExpectedRevenue"), F.lit(-2.0))) > 0.01)
    | (F.abs(F.coalesce(F.col("actual.AverageOrderValue"), F.lit(-1.0))
             - F.coalesce(F.col("expected.ExpectedAOV"), F.lit(-2.0))) > 0.01)
)
assert store_reconciliation.count() == 0

# Store totals must reconcile with the company order table.
assert store_performance.agg(F.sum("CompletedOrders")).first()[0] == orders_gold.count()
store_revenue = store_performance.agg(F.round(F.sum("NetRevenue"), 2)).first()[0]
order_revenue = orders_gold.agg(F.round(F.sum("NetRevenue"), 2)).first()[0]
assert abs(store_revenue - order_revenue) < 0.01


## 3. Display the observed business result and success marker


In [ ]:
top_store = store_performance.orderBy(F.desc("NetRevenue")).first()
print("Top store:", top_store.asDict())
display(
    store_performance.orderBy(F.desc("NetRevenue")).select(
        "StoreName", "CompletedOrders", "UniqueCustomers",
        "NetRevenue", "AverageOrderValue", "OnTimeRatePct"
    ).limit(10)
)

print("MANNAN_STORE_VALIDATION_PASSED")


## What the owner must be able to explain

- Which tables were compared and why.
- What each assertion protects against.
- What the displayed result means for FreshRoute.
- Why the final success marker `MANNAN_STORE_VALIDATION_PASSED` only prints after every check passes.
